# NFG Experiment 1B: Best PNE and POS **without** Weaker VEST Cut (Layered Graphs)

Same as Experiment 1, but with `add_vest_cut=False` in the GZR call.
This measures the computational impact of the weaker VEST cut
(fundamental cycle basis, Lemma 2 / Eq. 22) on the lazy CEI procedure.

**IMPORTANT**: Run this notebook in a **fresh kernel** after shutting down
notebook `02_run_nfg_layered.ipynb` to avoid execution order bias.

In [ ]:
import numpy as np
import pandas as pd
import time
from pathlib import Path

from gipg.nfg.instance import NFGInstance
from gipg.nfg.objectives import player_cost, all_player_costs, social_cost, edge_loads
from gipg.nfg.best_response import solve_best_response
from gipg.nfg.gzr import solve_gzr
from gipg.nfg.heuristics import alpha_of_profile, brd_random_restart
from gipg.nfg.social_optimum import solve_social_optimum, compute_pos

RESULTS_DIR = Path('../results')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print('NFG modules loaded successfully.')

## 1. Load Shared Layered Instances

In [ ]:
import json, glob

SHARED_DIR = Path('../data/nfg')
assert SHARED_DIR.exists(), f'Shared instances not found at {SHARED_DIR}. Run 01_generate_layered_networks.ipynb first.'

# Load only layered instances (Set E, F)
json_files = sorted(glob.glob(str(SHARED_DIR / 'E_*.json'))) + \
             sorted(glob.glob(str(SHARED_DIR / 'F_*.json')))
print(f'Found {len(json_files)} layered network files')

instances = {}
n_loaded = 0

for fp in json_files:
    with open(fp) as f:
        d = json.load(f)

    tag = d['tag']
    edges_tuple = tuple(tuple(e) for e in d['edges'])

    inst = NFGInstance(
        n_nodes=d['n_nodes'],
        edges=edges_tuple,
        n_players=d['n_players'],
        sources=np.array(d['sources'], dtype=int),
        sinks=np.array(d['sinks'], dtype=int),
        demands=np.array(d['demands'], dtype=int),
        capacities=np.array(d['capacities'], dtype=int),
        edge_costs=np.array(d['edge_costs'], dtype=float),
        seed=d['seed'],
        meta=d['meta'],
    )
    instances[tag] = inst
    n_loaded += 1

print(f'Loaded {n_loaded} NFG instances (layered graphs)')

SET_LABELS = {'E': 'tight+asym (layered)', 'F': 'loose+asym (layered)'}
for label, desc in SET_LABELS.items():
    count = sum(1 for t in instances if t.startswith(label + '_'))
    print(f'  Set {label} ({desc}): {count}')

## 2. Experiment 1B: Best PNE and POS without Weaker VEST Cut

For each instance:
1. **BRD** (random-restart best-response dynamics)
2. **GZR @ alpha=1** with `add_vest_cut=False`, `stop_at_first=False`
3. **Social Optimum** computed independently
4. **POS** = best_pne_cost / so_cost

In [ ]:
TOTAL_TIME_LIMIT = 600.0  # Total budget for BRD + GZR combined
SO_TIME_LIMIT = 1200.0    # 20 minutes for social optimum (reference computation)

exp1b_path = RESULTS_DIR / 'nfg_layered_exp1b_no_vest.csv'

results_exp1b = []

for idx, (tag, inst) in enumerate(instances.items()):
    cap_level = inst.meta.get('capacity_level', 'unknown')
    sym = inst.meta.get('symmetry', 'unknown')

    row = {
        'tag': tag,
        'graph_type': inst.meta.get('graph_type', 'layered'),
        'capacity_level': cap_level,
        'symmetry': sym,
        'n_players': inst.n_players,
        'n_nodes': inst.n_nodes,
        'n_edges': inst.n_edges,
        'n_layers': inst.meta.get('n_layers', None),
        'n_width': inst.meta.get('n_width', None),
        'seed': inst.meta.get('seed', None),
    }

    # --- Phase 1: BRD ---
    x_brd, brd_pne, brd_time = brd_random_restart(
        inst, max_init=3, max_round=15, seed=0,
    )
    row['brd_found_pne'] = brd_pne
    row['brd_time'] = brd_time
    row['brd_alpha'] = alpha_of_profile(inst, x_brd) if brd_pne else float('inf')
    if brd_pne:
        row['initial_pne_cost'] = social_cost(inst, x_brd)

    # --- Phase 2: GZR @ alpha=1, WITHOUT weaker VEST cut ---
    gzr_time_limit = max(TOTAL_TIME_LIMIT - brd_time, 60.0)

    warm = x_brd if brd_pne else None
    gzr_res = solve_gzr(
        inst, alpha=1.0, time_limit=gzr_time_limit,
        warm_start=warm,
        stop_at_first=False, verbose=False,
        add_vest_cut=False,  # <-- VEST off for Exp 1B
    )
    row['gzr_status'] = gzr_res.status
    row['gzr_mip_gap'] = gzr_res.mip_gap
    row['gzr_obj_bound'] = gzr_res.obj_bound
    row['gzr_cuts'] = gzr_res.cuts_added
    row['gzr_br_calls'] = gzr_res.br_calls
    row['gzr_time'] = brd_time + gzr_res.runtime
    row['gzr_obj_val'] = gzr_res.obj_val
    row['gzr_first_pne_time'] = (brd_time + gzr_res.first_pne_time
                                        if gzr_res.first_pne_time is not None else None)

    if gzr_res.profile is not None:
        row['best_pne_cost'] = social_cost(inst, gzr_res.profile)
    elif brd_pne:
        row['best_pne_cost'] = row['initial_pne_cost']
    else:
        row['best_pne_cost'] = None

    # --- Phase 3: Social Optimum (computed independently) ---
    so_res = solve_social_optimum(inst, time_limit=SO_TIME_LIMIT, verbose=False)
    row['so_status'] = so_res.status
    row['so_cost'] = so_res.opt_cost
    row['so_time'] = so_res.runtime

    # --- POS ---
    row['pos'] = None
    if row['best_pne_cost'] is not None and row['so_cost'] is not None and row['so_cost'] > 0:
        row['pos'] = compute_pos(row['best_pne_cost'], row['so_cost'])

    results_exp1b.append(row)

    # Progress
    pne_str = 'BRD-PNE' if brd_pne else 'no-PNE'
    pos_str = f"POS={row['pos']:.3f}" if row['pos'] is not None else 'POS=N/A'
    fpne_str = f"1stPNE={row['gzr_first_pne_time']:.1f}s" if row.get('gzr_first_pne_time') is not None else '1stPNE=N/A'
    print(f'[{idx+1}/{len(instances)}] {tag}: {pne_str} | GZR(noVEST) {gzr_res.status} '
          f'({row["gzr_time"]:.1f}s, {gzr_res.cuts_added}cuts) | {pos_str} | {fpne_str}')

df_exp1b = pd.DataFrame(results_exp1b)
df_exp1b.to_csv(exp1b_path, index=False)
print(f'\nSaved {len(df_exp1b)} rows to {exp1b_path.name}')

## 3. Summary

In [ ]:
print('=== Experiment 1B Summary (Layered NFG, no weaker VEST) ===\n')

n_total = len(df_exp1b)
n_brd_pne = df_exp1b['brd_found_pne'].sum()
n_gzr_opt = (df_exp1b['gzr_status'] == 'OPTIMAL').sum()
n_gzr_inf = (df_exp1b['gzr_status'] == 'INFEASIBLE').sum()
n_gzr_tl = (df_exp1b['gzr_status'] == 'TIME_LIMIT').sum()
n_pos = df_exp1b['pos'].notna().sum()

print(f'Total instances: {n_total}')
print(f'BRD found PNE: {n_brd_pne} ({n_brd_pne/n_total:.1%})')
print(f'GZR OPTIMAL (no VEST): {n_gzr_opt} ({n_gzr_opt/n_total:.1%})')
print(f'GZR INFEASIBLE: {n_gzr_inf} ({n_gzr_inf/n_total:.1%})')
print(f'GZR TIME_LIMIT: {n_gzr_tl} ({n_gzr_tl/n_total:.1%})')
print(f'POS computed: {n_pos} ({n_pos/n_total:.1%})')

# By capacity_level
print('\n--- By capacity_level ---')
summary = df_exp1b.groupby(['capacity_level']).agg(
    n=('tag', 'count'),
    brd_pne_rate=('brd_found_pne', 'mean'),
    gzr_opt_rate=('gzr_status', lambda x: (x == 'OPTIMAL').mean()),
    gzr_inf_rate=('gzr_status', lambda x: (x == 'INFEASIBLE').mean()),
    avg_gzr_time=('gzr_time', 'mean'),
    avg_cuts=('gzr_cuts', 'mean'),
    avg_pos=('pos', 'mean'),
    max_pos=('pos', 'max'),
).round(4)
print(summary)

# By (n_layers, n_width)
print('\n--- By (n_layers, n_width) ---')
summary_lw = df_exp1b.groupby(['n_layers', 'n_width']).agg(
    n=('tag', 'count'),
    gzr_opt_rate=('gzr_status', lambda x: (x == 'OPTIMAL').mean()),
    avg_gzr_time=('gzr_time', 'mean'),
    avg_cuts=('gzr_cuts', 'mean'),
    avg_pos=('pos', 'mean'),
).round(4)
print(summary_lw)